Ramalah Amir </br>
23i-2644 </br>
DS-6C </br>


# Loading the Dataset


In [ ]:
import gzip
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random
import re
from collections import Counter
from torch.utils.data import Dataset, DataLoader
import os

random.seed(42)
torch.manual_seed(42)


def load_data(file_path, max_samples=15000):
    data = []

    with gzip.open(file_path, "rt", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= max_samples:
                break
            record = json.loads(line)

            if "reviewText" in record and "overall" in record:
                data.append(
                    {"review": record["reviewText"], "rating": record["overall"]}
                )
    return data


cat1 = load_data("Arts_Crafts_and_Sewing_5.json.gz", 15000)
cat2 = load_data("Digital_Music_5.json.gz", 15000)
cat3 = load_data("Movies_and_TV_5.json.gz", 15000)

all_data = cat1 + cat2 + cat3
random.shuffle(all_data)
print(f"Total samples loaded: {len(all_data)}")

Total samples loaded: 44980


# Extracting Targets


In [ ]:
def process_samples(data):
    processed = []
    for d in data:
        text = str(d["review"]).lower()
        rating = d["rating"]

        # Target 01: sentiment classification
        # 1-2 -> Negative (0), 3 -> Neutral (1), 4-5 -> Positive (2)
        if rating <= 2:
            sentiment = 0
        elif rating == 3:
            sentiment = 1
        else:
            sentiment = 2

        # Target 02: Length Classification
        length = len(text.split())
        if length < 20:
            length_class = 0  # Short
        elif length <= 50:
            length_class = 1  # Medium
        else:
            length_class = 2  # Long

        processed.append(
            {"text": text, "sentiment": sentiment, "length_class": length_class}
        )
    return processed


processed_data = process_samples(all_data)

# Split 70% Train, 15% Val, 15% Test
n = len(processed_data)
train_end = int(0.7 * n)
val_end = int(0.85 * n)

train_data = processed_data[:train_end]
val_data = processed_data[train_end:val_end]
test_data = processed_data[val_end:]

print(f"Train samples: {len(train_data)}")
print(f"Val samples:   {len(val_data)}")
print(f"Test samples:  {len(test_data)}")